# OLaLa on Google Colab A100

Use an A100 GPU runtime with High-RAM enabled. This notebook uses `Qwen/Qwen2.5-7B-Instruct`, which avoids the gated-access error from Meta Llama models.

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk maven
!java -version
!mvn -version
!nvidia-smi

In [ ]:
%cd /content
!rm -rf melt
!git clone -b main https://github.com/Phearun2020/melt.git
%cd /content/melt

In [ ]:
!python -m pip install -U "pip<26" "setuptools<81" wheel
!python -m pip uninstall -y torchvision torchaudio torchtext
!python -m pip install -U --force-reinstall --no-cache-dir \
  "numpy>=1.26,<2.0" \
  "scipy==1.12.0" \
  "pandas>=2.2,<2.3" \
  "scikit-learn>=1.5,<1.6" \
  "gensim>=4.4.0" \
  "flask>=2.0" \
  "Werkzeug<=2.2.3" \
  "protobuf>=5.28,<6"
!python -m pip install -U --force-reinstall --no-cache-dir --upgrade-strategy only-if-needed \
  "sentence-transformers==2.7.0" "transformers==4.41.2" "huggingface-hub>=0.23,<1" \
  accelerate bitsandbytes sentencepiece
!python -m pip uninstall -y torchvision torchaudio torchtext

In [ ]:
import os

print("Restarting the Colab runtime so NumPy/SciPy/Transformers reload cleanly.")
print("After Colab reconnects, continue from the Hugging Face login cell below.")
os.kill(os.getpid(), 9)

After Colab reconnects, continue here. Do not rerun the dependency installation cell unless you also rerun the restart cell.

In [ ]:
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
import google.protobuf
import torch

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
AutoTokenizer.from_pretrained(MODEL_ID)
SentenceTransformer("multi-qa-mpnet-base-dot-v1", device="cuda" if torch.cuda.is_available() else "cpu")
print(f"{MODEL_ID} access OK")
print(f"Protobuf: {google.protobuf.__version__}")
print(f"Torch: {torch.__version__}; CUDA available: {torch.cuda.is_available()}")

In [ ]:
%cd /content/melt
!grep -n "babelnet-offline" matching-jena-matchers/pom.xml
!grep -n "Qwen\\|device_map\\|load_in_4bit\\|bnb_4bit" examples/llm-transformers/src/main/java/de/uni_mannheim/informatik/dws/melt/examples/llm_transformers/OLaLaForOAEI.java

In [ ]:
%cd /content/melt
!mvn -U -pl matching-eval,matching-ml,receiver-http,matching-owlapi-matchers -am install -DskipTests -Dmaven.test.skip=true -Dmaven.javadoc.skip=true

In [ ]:
%cd /content/melt/examples/llm-transformers
!mvn -U clean package -DskipTests

In [ ]:
%cd /content/melt/examples/llm-transformers
!CUDA_VISIBLE_DEVICES=0 java -Xmx40G -jar target/llm-transformers-1.0.jar --gpu 0 --limit 5